In [2]:
import pandas as pd
import numpy as np

from pprint import pprint

# models
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV


In [3]:
df = pd.read_csv(
    filepath_or_buffer="work_from_home_burnout_dataset.csv"
)

In [4]:
df

,user_id,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,burnout_score,burnout_risk
0,1,Weekday,9.59,11.86,4,2,0,7.55,91.2,19.17,Low
1,1,Weekend,7.38,10.33,4,1,0,6.69,82.0,29.70,Low
2,1,Weekend,6.31,8.92,1,2,0,8.87,80.6,32.93,Low
3,1,Weekday,8.34,10.70,4,1,1,8.13,70.0,45.47,Low
4,1,Weekend,6.97,9.83,1,2,0,5.85,67.1,51.61,Low
...,...,...,...,...,...,...,...,...,...,...,...
1795,180,Weekend,6.33,8.16,0,4,0,5.59,73.5,31.91,Low
1796,180,Weekend,4.70,7.88,0,4,0,6.69,89.8,26.30,Low
1797,180,Weekend,3.92,6.39,2,1,0,6.77,74.6,34.07,Low
1798,180,Weekday,8.93,11.11,2,5,0,8.28,74.6,38.14,Low


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   user_id               1800 non-null   int64  
 1   day_type              1800 non-null   object 
 2   work_hours            1800 non-null   float64
 3   screen_time_hours     1800 non-null   float64
 4   meetings_count        1800 non-null   int64  
 5   breaks_taken          1800 non-null   int64  
 6   after_hours_work      1800 non-null   int64  
 7   sleep_hours           1800 non-null   float64
 8   task_completion_rate  1800 non-null   float64
 9   burnout_score         1800 non-null   float64
 10  burnout_risk          1800 non-null   object 
dtypes: float64(5), int64(4), object(2)
memory usage: 154.8+ KB


In [6]:
df["day_type"].value_counts()

day_type
Weekend    924
Weekday    876
Name: count, dtype: int64

In [7]:
mapper = {
    "Weekend": 0,
    "Weekday": 1
}

df["day_type"] = df["day_type"].map(mapper)

In [8]:
df["day_type"].value_counts()

day_type
0    924
1    876
Name: count, dtype: int64

I will firstly try to work with `burnout_risk` and then pay attention to `burnout_score`.

In [9]:
records_table = pd.DataFrame(columns=["model", "accuracy"])
records_table

,model,accuracy


### Gradient Boosting Classifier

In [10]:
X = df.drop(
    columns=["user_id", "burnout_score", "burnout_risk"],
)

y = df["burnout_risk"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35
)

In [12]:
X.head()

,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate
0,1,9.59,11.86,4,2,0,7.55,91.2
1,0,7.38,10.33,4,1,0,6.69,82.0
2,0,6.31,8.92,1,2,0,8.87,80.6
3,1,8.34,10.70,4,1,1,8.13,70.0
4,0,6.97,9.83,1,2,0,5.85,67.1


In [13]:
y.head()

0    Low
1    Low
2    Low
3    Low
4    Low
Name: burnout_risk, dtype: object

In [14]:
gradient_boosting_classifier = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

In [15]:
gradient_boosting_classifier.fit(X_train, y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [28]:
y_pred = gradient_boosting_classifier.predict(X_test)
gbc_accuracy = np.round(accuracy_score(y_test, y_pred), 2)

In [29]:
print("Accuracy:", gbc_accuracy)

Accuracy: 0.95


In [30]:
records_table.loc[0, "model"] = 'Gradient Boosting Classifier'
records_table.loc[0, "accuracy"] = gbc_accuracy

In [31]:
records_table

,model,accuracy
0,Gradient Boosting Classifier,0.95


### Randomized Search of Best Hyperparameters

In [18]:
model = GradientBoostingClassifier()

In [19]:
param_dict = {
    "n_estimators": np.arange(100, 500, step=25),
    "learning_rate": np.linspace(0.01, 0.2, num=25),
    "max_depth": np.arange(3, 8)
}

random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dict,
    n_iter=30,                       # number of random combos
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=2
)

In [20]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


KeyboardInterrupt: 

In [ ]:
print("Best parameters:", random_search.best_params_)

Best parameters: {'n_estimators': np.int64(325), 'max_depth': np.int64(3), 'learning_rate': np.float64(0.1366666666666667)}


In [ ]:
y_pred = random_search.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9492063492063492


### Exhaustive Search of Best Hyperparameters (GridSearchCV)

In [ ]:
model = GradientBoostingClassifier()

In [ ]:
param_grid = {
    "n_estimators": np.arange(100, 500, step=25),
    "learning_rate": np.linspace(0.1, 2, num=25),
    "max_depth": np.arange(3, 8)
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,                 # 5-fold cross validation
    scoring="accuracy",   # metric
    n_jobs=-1,            # use all CPU cores
    verbose=2
)

In [ ]:
# grid.fit(X_train, y_train)

In [ ]:
# print("Best parameters:", grid.best_params_)
# print("Best CV score:", grid.best_score_)

In [ ]:
# y_pred = grid.predict(X_test)
# print("Accuracy:", accuracy_score(y_test, y_pred))